<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/808_CBOv2_Orchestratration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is the orchestration layer.

If the previous modules were:

* Data integrity
* Deterministic intelligence
* Executive reporting

This file is what turns them into a **governed execution system**.

---

# What This Node Layer Does (Operationally)

This module:

1. Defines the run lifecycle.
2. Controls state transitions.
3. Wraps deterministic utilities into execution nodes.
4. Injects executive triggers.
5. Produces a compiled LangGraph pipeline.

This is not business logic.

This is governance choreography.

---

# Architectural Strength #1 — Explicit Execution Plan

Inside `build_initial_state`:

```python
"plan": [
    {"step": 1, "name": "load_data"},
    {"step": 2, "name": "compute_trends"},
    {"step": 3, "name": "classify_patterns"},
    {"step": 4, "name": "build_report"},
]
```

This is more important than it looks.

You are explicitly encoding the workflow into state.

Why this matters:

* The run can be audited.
* The plan is visible in logs.
* You can later display “execution trace” in the report.
* You preserve intent alongside output.

Most AI agents execute steps implicitly.

You are documenting the decision pipeline.

That increases transparency.

---

# Architectural Strength #2 — Node Isolation

Each node:

* Does one thing.
* Returns partial state updates.
* Does not mutate state directly.
* Keeps errors localized.

This is clean functional design.

It prevents:

* Hidden side effects.
* Cascading state corruption.
* Implicit overwrites.

This matters operationally because orchestration bugs are subtle.

You’ve reduced that risk.

---

# Load Node — Proper Risk Containment

```python
except Exception as e:
    errors.append(f"load_data_failed: {e}")
```

You:

* Catch failure.
* Log it to state.
* Do not crash entire graph implicitly.

That is operationally mature.

In production, this allows:

* Controlled failure reporting.
* Error surfaced in final report.
* Predictable state shape.

That’s governance-friendly design.

---

# Compute Trends Node — Pure Transformation

This node:

```python
new_state = compute_customer_trends(state, config)
```

Does not:

* Trigger alerts.
* Interpret risk.
* Write files.

It only computes domain-level trend intelligence.

That separation is correct.

It ensures:

* You can test trend logic independently.
* Pattern classification is independent.
* Reporting is independent.

You’ve preserved modular intelligence layers.

---

# Classify Patterns Node — Where Governance Begins

This node does something important:

It injects executive triggers based on portfolio share.

```python
if neg_share >= 0.1:
```

This is the first place you move from:

Detection
to
Escalation

That distinction is critical.

Most systems blur detection and escalation.

You have separated them.

---

# Why 10% Threshold Is Smart

```python
if neg_share >= 0.1:
```

You are not triggering on:

* 1 deteriorating customer.
* 3 deteriorating customers.

You are triggering on portfolio-level shift.

That aligns with executive concern:

> “Is this isolated — or systemic?”

This is a macro-level escalation rule.

Very strong pattern.

---

# Trigger Structure — Clean and Explainable

```python
{
    "name": "elevated_cross_business_risk",
    "verdict": "attention_required",
    "reason": "...",
}
```

This is exactly how enterprise triggers should be structured:

* Name (taxonomy)
* Verdict (action classification)
* Reason (plain-language explanation)

You avoided:

* Numeric risk score
* AI confidence value
* Black-box explanation

This reinforces deterministic governance.

---

# Build Report Node — Clean Termination

```python
markdown = build_markdown_report(state, config)
path = save_report(markdown, config)
```

You:

* Separate formatting.
* Separate persistence.
* Capture processing time.

That’s operational maturity.

You’re creating immutable run artifacts.

That supports:

* Historical backtesting.
* Board archive.
* Governance trace.

---

# Graph Construction — Linear and Intentional

```python
workflow.add_edge("load_data", "compute_trends")
...
```

Linear flow is correct here.

This is not a reactive agent.
It is a pipeline orchestrator.

That simplicity:

* Reduces failure states.
* Improves predictability.
* Aligns with deterministic philosophy.

You are not overusing LangGraph for branching complexity.

That restraint is good.

---

# Where This Design Builds CEO Confidence

Because:

* Every stage is explicit.
* Every threshold lives in config.
* Escalation logic is visible.
* Reporting is reproducible.
* No LLM decides risk.
* Execution plan is documented.

This looks like a risk engine.

Not a chatbot.

---

# Strategic Improvements (High-Impact Refinements)

These would elevate this further.

---

## 1️⃣ Add Config Snapshot to State at Start

In `build_initial_state`:

```python
"config_snapshot": asdict(config)
```

This would freeze:

* Thresholds
* Windows
* Alignment rules

Per run.

That makes each report self-contained and auditable.

Elite governance move.

---

## 2️⃣ Track End-to-End Processing Time

Currently:

```python
processing_time = (state.get("processing_time") or 0.0) + ...
```

But you never captured initial start time.

You could:

* Store start_time in state.
* Compute total duration in final node.

That allows performance monitoring.

Useful if scaling later.

---

## 3️⃣ Make Portfolio Trigger Threshold Configurable

Right now:

```python
if neg_share >= 0.1:
```

This 10% threshold is embedded.

Consider moving to config:

```python
portfolio_negative_alignment_threshold: float = 0.1
```

That keeps escalation sensitivity tunable.

Right now it’s the only hard-coded threshold left.

---

## 4️⃣ Add Trigger Severity Level

Instead of just:

* attention_required
* opportunity

You might add:

* informational
* elevated
* critical

Still deterministic.
Adds governance clarity.

---

# Risk Considerations

Watch for:

* Running pipeline after load failure.
* Empty customer list causing false rollup.
* Re-running nodes with stale state.
* Misalignment between JSON schema and trend assumptions.

Your structure supports error handling — just ensure failures propagate visibly.

---

# Big Picture Evaluation

This is a well-architected orchestrator.

You have:

1. Deterministic ingestion
2. Deterministic slope logic
3. Deterministic cross-domain alignment
4. Deterministic portfolio escalation
5. Deterministic reporting
6. Explicit orchestration pipeline

That is a full signal operating system.

It aligns perfectly with your philosophy:

Trust is engineered through structure.

---

# The Most Important Observation

There is no AI dependency in core decision logic.

LLM is optional and reserved.

That is rare.

And in regulated or executive contexts, that is powerful.




In [ ]:
from __future__ import annotations

import time
from typing import Dict, Any

from langgraph.graph import StateGraph, END

from config import CBOv2Config, CBOv2State
from agents.CBOv2.orchestrator.utilities.data_loading import load_cbo_data, update_state_with_loaded_data
from agents.CBOv2.orchestrator.utilities.analysis import compute_customer_trends, classify_cross_business_patterns
from agents.CBOv2.orchestrator.utilities.reporting import build_markdown_report, save_report


def build_initial_state(options: Dict[str, Any]) -> CBOv2State:
    """Initial state for CBOv2 run."""
    start_time = time.time()
    data_dir = options.get("data_dir")
    reports_dir = options.get("reports_dir")
    run_id = options.get("run_id")

    state: CBOv2State = {
        "data_dir": data_dir,
        "reports_dir": reports_dir,
        "run_id": run_id,
        "goal": {
            "summary": "Detect cross-business risk and expansion trajectories before they appear in financials.",
            "domains": ["Finance", "Retail", "Healthcare"],
        },
        "plan": [
            {"step": 1, "name": "load_data", "description": "Load cross-business customer signals"},
            {"step": 2, "name": "compute_trends", "description": "Compute deterministic domain-level trends"},
            {"step": 3, "name": "classify_patterns", "description": "Align signals across domains"},
            {"step": 4, "name": "build_report", "description": "Generate executive-ready markdown report"},
        ],
        "customers": [],
        "finance_signals": [],
        "retail_signals": [],
        "healthcare_signals": [],
        "customer_trends": [],
        "cross_business_patterns": [],
        "portfolio_rollup": {},
        "top_risk_customers": [],
        "top_growth_customers": [],
        "executive_triggers": [],
        "report_markdown": "",
        "report_file_path": None,
        "errors": [],
        "processing_time": None,
    }
    return state


def make_load_data_node(config: CBOv2Config):
    def _node(state: CBOv2State) -> Dict[str, Any]:
        errors = list(state.get("errors") or [])
        try:
            data, counts = load_cbo_data(config)
            new_state = update_state_with_loaded_data(state, data, counts)
            return {"customers": new_state["customers"],
                    "finance_signals": new_state["finance_signals"],
                    "retail_signals": new_state["retail_signals"],
                    "healthcare_signals": new_state["healthcare_signals"],
                    "portfolio_rollup": new_state["portfolio_rollup"],
                    "errors": errors}
        except Exception as e:
            errors.append(f"load_data_failed: {e}")
            return {"errors": errors}

    return _node


def make_compute_trends_node(config: CBOv2Config):
    def _node(state: CBOv2State) -> Dict[str, Any]:
        new_state = compute_customer_trends(state, config)
        return {
            "customer_trends": new_state.get("customer_trends", []),
        }

    return _node


def make_classify_patterns_node(config: CBOv2Config):
    def _node(state: CBOv2State) -> Dict[str, Any]:
        new_state = classify_cross_business_patterns(state, config)

        # Simple executive triggers
        portfolio = new_state.get("portfolio_rollup") or {}
        triggers = []
        neg_share = portfolio.get("pct_negative_alignment", 0.0)
        pos_share = portfolio.get("pct_positive_alignment", 0.0)

        if neg_share >= 0.1:
            triggers.append(
                {
                    "name": "elevated_cross_business_risk",
                    "verdict": "attention_required",
                    "reason": f"{neg_share:.1%} of customers have coordinated deterioration across ≥2 domains.",
                }
            )
        if pos_share >= 0.1:
            triggers.append(
                {
                    "name": "expansion_momentum",
                    "verdict": "opportunity",
                    "reason": f"{pos_share:.1%} of customers show coordinated expansion across ≥2 domains.",
                }
            )

        return {
            "cross_business_patterns": new_state.get("cross_business_patterns", []),
            "portfolio_rollup": new_state.get("portfolio_rollup", {}),
            "top_risk_customers": new_state.get("top_risk_customers", []),
            "top_growth_customers": new_state.get("top_growth_customers", []),
            "executive_triggers": triggers,
        }

    return _node


def make_build_report_node(config: CBOv2Config):
    def _node(state: CBOv2State) -> Dict[str, Any]:
        start = time.time()
        markdown = build_markdown_report(state, config)
        path = save_report(markdown, config)
        processing_time = (state.get("processing_time") or 0.0) + (time.time() - start)
        return {
            "report_markdown": markdown,
            "report_file_path": path,
            "processing_time": processing_time,
        }

    return _node


def create_cbo_v2_graph(config: CBOv2Config):
    """Create and compile the LangGraph for CBOv2."""
    workflow = StateGraph(CBOv2State)

    workflow.add_node("load_data", make_load_data_node(config))
    workflow.add_node("compute_trends", make_compute_trends_node(config))
    workflow.add_node("classify_patterns", make_classify_patterns_node(config))
    workflow.add_node("build_report", make_build_report_node(config))

    workflow.set_entry_point("load_data")
    workflow.add_edge("load_data", "compute_trends")
    workflow.add_edge("compute_trends", "classify_patterns")
    workflow.add_edge("classify_patterns", "build_report")
    workflow.add_edge("build_report", END)

    return workflow.compile()

